# Segmentación de Clientes y Predicción de Churn — Entrega 03

**Ciencia de Datos Aplicada · ITBA · 1er Cuatrimestre 2026 · Grupo 12**

Notebook unificado de **modelado** (Entrega 03). Cuenta, de punta a punta, cómo pasamos de los datos de clientes a dos modelos que generan valor de negocio:

1. **Segmentación de clientes** — aprendizaje no supervisado (K-Means).
2. **Predicción de churn** — aprendizaje supervisado (Random Forest).

> **Objetivo de la entrega:** describir el enfoque y su justificación, implementar los modelos, evaluarlos con métricas apropiadas, analizar los resultados, reflexionar críticamente y dejar los modelos **persistidos y reutilizables sin reentrenar**.

## 1. El problema de negocio

Trabajamos con el dataset **Online Retail** (UCI): un retailer online del Reino Unido con **397.884 transacciones válidas** y **4.338 clientes** únicos.

La empresa enfrenta dos preguntas de negocio:

- **¿Qué tipos de clientes tenemos?** No todos valen ni se comportan igual. Necesitamos agruparlos para accionar distinto en cada grupo (retener, hacer crecer, reactivar).
- **¿Quiénes están por irse?** Retener a un cliente es mucho más barato que adquirir uno nuevo. Necesitamos anticipar el abandono para actuar a tiempo.

Dos preguntas → **dos modelos complementarios**.

## 2. Enfoque adoptado y justificación

| Pregunta | Técnica | Por qué |
|----------|---------|---------|
| ¿Qué clientes tenemos? | **K-Means** (no supervisado) sobre RFM + atributos de producto | Agrupa clientes por comportamiento sin necesidad de etiquetas; habilita estrategias diferenciadas por segmento. |
| ¿Quiénes se van a ir? | **Random Forest** (supervisado) | Modelo robusto y poco sensible a outliers, interpretable vía importancia de variables. |

**Definición de churn (sin data leakage):** dividimos la línea de tiempo en un período de *observación* y uno de *predicción*. Un cliente es churn (`1`) si compró en observación pero **no** volvió a comprar en el período de predicción. Las variables se calculan únicamente con datos del período de observación.

Ambos modelos se guardan en formato reutilizable (`.pkl`) para poder usarlos luego sin reentrenar.

In [1]:
import json
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    silhouette_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

sns.set_theme(style="whitegrid")

CHURN_THRESHOLD = 0.5

_candidates = [Path("../../data"), Path("data"), Path("../data")]
DATA_DIR = next((c for c in _candidates if c.exists()), Path("../../data"))
MODELS_DIR = DATA_DIR / "06_models"

rfm = pd.read_parquet(DATA_DIR / "04_feature/rfm_clientes_enriched.parquet")
churn = pd.read_parquet(DATA_DIR / "05_model_input/churn_dataset.parquet")

print(f"Clientes para segmentación: {len(rfm):,}")
print(f"Clientes para churn (con ventana temporal): {len(churn):,}")
print(f"Churn rate observado: {churn['Churn'].mean():.1%}")

Clientes para segmentación: 4,338
Clientes para churn (con ventana temporal): 3,317
Churn rate observado: 41.2%


## 3. Segmentación de clientes (K-Means)

Usamos **8 variables**: RFM (Recency, Frequency, Monetary) más atributos de afinidad de producto derivados del enriquecimiento de la Entrega 02.

> **Mejora post-feedback (Entrega 03):** la devolución señaló un silhouette bajo (~0.17), indicando clusters poco separados. Revisamos *features* y método: (1) **excluimos `Cancel_rate`** del clustering —tiene 64% de ceros y outliers extremos, por lo que aportaba ruido; se conserva como feature del modelo de churn—, y (2) agregamos **PCA** (reducción de dimensionalidad, U4) antes de K-Means para de-correlacionar y compactar el espacio. El silhouette a k=4 sube de ~0.17 a **~0.31 (+80%)**, manteniendo la afinidad de producto.

Detalle técnico clave: el preprocesamiento (`log1p` en variables asimétricas + estandarización + PCA) va dentro de un **`Pipeline`** que se serializa junto al modelo. Así la inferencia reproduce exactamente el entrenamiento y el artefacto es autocontenido.

Elegimos **k = 4** para preservar los 4 segmentos de negocio interpretables.

In [2]:
FEATURES_CLUSTER = [
    "Recency",
    "Frequency",
    "Monetary",
    "pct_with_color",
    "color_diversity",
    "pct_with_material",
    "avg_quantity_in_set",
    "pct_purchases_sets",
]
LOG_FEATURES = ["Frequency", "Monetary"]
OTHER_FEATURES = [f for f in FEATURES_CLUSTER if f not in LOG_FEATURES]

# Pipeline: log1p (asimétricas) -> estandarización -> PCA (reducción de dimensionalidad).
# PCA de-correlaciona y compacta el espacio, mejorando la separación de clusters.
preprocess = Pipeline(
    [
        (
            "log",
            ColumnTransformer(
                [
                    (
                        "log1p",
                        FunctionTransformer(np.log1p, feature_names_out="one-to-one"),
                        LOG_FEATURES,
                    ),
                    ("passthrough", "passthrough", OTHER_FEATURES),
                ]
            ),
        ),
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=3, random_state=42)),
    ]
)

X_cluster = preprocess.fit_transform(rfm[FEATURES_CLUSTER])
kmeans = KMeans(n_clusters=4, random_state=42, n_init=20)
rfm["Cluster"] = kmeans.fit_predict(X_cluster)

# Etiquetas de negocio derivadas del perfil de cada cluster (robusto al número de cluster).
# Cancel_rate ya no es feature de clustering, pero se usa como descriptor para etiquetar
# el segmento "En Riesgo".
prof = rfm.groupby("Cluster").agg(
    monetary=("Monetary", "mean"),
    recency=("Recency", "mean"),
    cancel=("Cancel_rate", "mean"),
    sets=("pct_purchases_sets", "mean"),
)
labels = {}
labels[prof["monetary"].idxmax()] = "VIP"
rest = prof.drop(index=list(labels))
labels[rest["cancel"].idxmax()] = "En Riesgo"
rest = prof.drop(index=list(labels))
labels[rest["sets"].idxmax()] = "Compradores de Sets"
rest = prof.drop(index=list(labels))
labels[rest["recency"].idxmax()] = "Dormidos"

rfm["Segment_label"] = rfm["Cluster"].map(labels)
sil = silhouette_score(X_cluster, rfm["Cluster"])

print(f"Silhouette score: {sil:.3f}")
print("\nClientes por segmento:")
print(rfm["Segment_label"].value_counts())

Silhouette score: 0.314

Clientes por segmento:
Segment_label
VIP                    1787
Dormidos                981
En Riesgo               965
Compradores de Sets     605
Name: count, dtype: int64


In [3]:
# Perfil de negocio por segmento
total_rev = rfm["Monetary"].sum()
perfil = (
    rfm.groupby("Segment_label")
    .agg(
        clientes=("CustomerID", "count"),
        recency_media=("Recency", "mean"),
        frequency_media=("Frequency", "mean"),
        monetary_medio=("Monetary", "mean"),
        cancel_rate=("Cancel_rate", "mean"),
        revenue_total=("Monetary", "sum"),
    )
    .round(1)
)
perfil["% clientes"] = (perfil["clientes"] / perfil["clientes"].sum() * 100).round(1)
perfil["% revenue"] = (perfil["revenue_total"] / total_rev * 100).round(1)
perfil.sort_values("revenue_total", ascending=False)

,clientes,recency_media,frequency_media,monetary_medio,cancel_rate,revenue_total,% clientes,% revenue
Segment_label,,,,,,,,
VIP,1787,32.7,7.9,3934.9,2.2,7031614.2,41.2,84.6
En Riesgo,965,111.3,1.7,448.3,9.6,432620.9,22.2,5.2
Compradores de Sets,605,112.6,2.0,708.8,2.6,428846.0,13.9,5.2
Dormidos,981,170.7,1.5,428.4,2.4,420213.8,22.6,5.1


In [4]:
perfil_ord = perfil.sort_values("% revenue", ascending=True)
fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(perfil_ord.index, perfil_ord["% revenue"], color="steelblue")
ax.set_xlabel("% del revenue total")
ax.set_title("Concentración de revenue por segmento")
for bar, pct, cli in zip(bars, perfil_ord["% revenue"], perfil_ord["% clientes"], strict=False):
    ax.text(
        bar.get_width() + 1,
        bar.get_y() + bar.get_height() / 2,
        f"{pct:.1f}% rev · {cli:.1f}% clientes",
        va="center",
    )
ax.set_xlim(0, 100)
plt.tight_layout()
plt.show()

/var/folders/80/chm8f4xj689g2c7r85cswh3m0000gn/T/ipykernel_78440/2994935019.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


> **Insight de negocio (efecto Pareto):** el segmento **VIP** concentra la enorme mayoría del revenue pese a ser una fracción de la base de clientes. Perder un cliente VIP no equivale a perder uno promedio. Esto justifica tratar a los segmentos de forma distinta:
>
> - **VIP:** retención prioritaria y cuidado personalizado.
> - **Compradores de Sets:** cross-selling de packs y bundles.
> - **En Riesgo:** atacar fricciones y cancelaciones (su `cancel_rate` es el más alto).
> - **Dormidos:** campañas de reactivación de bajo costo.

## 4. Predicción de churn (Random Forest)

Entrenamos un clasificador para estimar la **probabilidad de abandono** de cada cliente, usando 14 variables (RFM + producto + actividad).

- **Split:** 80% entrenamiento / 20% test, estratificado por la clase.
- **Modelo:** Random Forest (`n_estimators=200`, `max_depth=10`, `min_samples_leaf=10`), seleccionado por F1 en validación cruzada frente a alternativas.
- **Métricas:** AUC-ROC, F1, precisión y recall sobre el conjunto de test.

In [5]:
FEATURE_NAMES = [
    "Recency",
    "Frequency",
    "Monetary",
    "Cancel_rate",
    "pct_with_color",
    "color_diversity",
    "is_color_specialist",
    "pct_with_material",
    "pct_purchases_sets",
    "avg_days_between_purchases",
    "months_active",
    "n_products_unique",
    "avg_order_value",
]

X = churn[FEATURE_NAMES].to_numpy()
y = churn["Churn"].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

rf = RandomForestClassifier(
    n_estimators=200, max_depth=10, min_samples_leaf=10, random_state=42, n_jobs=-1
)
rf.fit(X_train_scaled, y_train)

print(f"Train: {len(X_train):,} clientes · Test: {len(X_test):,} clientes")

Train: 2,653 clientes · Test: 664 clientes


In [6]:
y_prob = rf.predict_proba(X_test_scaled)[:, 1]
y_pred = rf.predict(X_test_scaled)

print("=== Métricas en test ===")
print(f"AUC-ROC  : {roc_auc_score(y_test, y_prob):.3f}")
print(f"F1-score : {f1_score(y_test, y_pred):.3f}")
print(f"Precisión: {precision_score(y_test, y_pred):.3f}")
print(f"Recall   : {recall_score(y_test, y_pred):.3f}")

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(4.5, 4))
ConfusionMatrixDisplay(cm, display_labels=["Retenido", "Churn"]).plot(
    cmap="Blues", ax=ax, colorbar=False
)
ax.set_title("Matriz de confusión (test)")
plt.tight_layout()
plt.show()

=== Métricas en test ===
AUC-ROC  : 0.744
F1-score : 0.623
Precisión: 0.608
Recall   : 0.637


/var/folders/80/chm8f4xj689g2c7r85cswh3m0000gn/T/ipykernel_78440/3749300250.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
importancias = pd.Series(rf.feature_importances_, index=FEATURE_NAMES).sort_values()
fig, ax = plt.subplots(figsize=(8, 5))
importancias.plot.barh(ax=ax, color="darkorange")
ax.set_title("¿Qué variables explican el churn?")
ax.set_xlabel("Importancia")
plt.tight_layout()
plt.show()

/var/folders/80/chm8f4xj689g2c7r85cswh3m0000gn/T/ipykernel_78440/3811279655.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


> **Insight de churn:** las variables de **recencia y frecuencia/recurrencia** dominan la predicción: un cliente que hace mucho que no compra y con baja cadencia es el principal candidato a abandono. El modelo no solo predice, también señala *dónde mirar* para diseñar acciones de retención.

## 5. Dónde está el revenue en riesgo (segmento × churn)

Acá los dos modelos se combinan y aparece el mayor valor de negocio: cruzamos el **segmento** de cada cliente con su **probabilidad de churn** para responder *no solo cuántos clientes se van, sino cuánto revenue está en juego y en qué segmento*.

In [8]:
# Scorear toda la base de churn y cruzar con el segmento y el valor del cliente
scored = churn[["CustomerID", "Churn", *FEATURE_NAMES]].copy()
scored["churn_prob"] = rf.predict_proba(scaler.transform(scored[FEATURE_NAMES].to_numpy()))[:, 1]
scored["churn_pred"] = (scored["churn_prob"] >= CHURN_THRESHOLD).astype(int)
scored = scored.merge(
    rfm[["CustomerID", "Segment_label", "Monetary"]].rename(columns={"Monetary": "valor_cliente"}),
    on="CustomerID",
    how="left",
)

en_riesgo = scored[scored["churn_pred"] == 1]
resumen = scored.groupby("Segment_label").agg(
    clientes=("CustomerID", "count"),
    churn_real_pct=("Churn", "mean"),
    churn_prob_media=("churn_prob", "mean"),
)
resumen["revenue_en_riesgo"] = en_riesgo.groupby("Segment_label")["valor_cliente"].sum()
resumen["churn_real_pct"] = (resumen["churn_real_pct"] * 100).round(1)
resumen["churn_prob_media"] = (resumen["churn_prob_media"] * 100).round(1)
resumen["revenue_en_riesgo"] = resumen["revenue_en_riesgo"].round(0)
resumen.sort_values("revenue_en_riesgo", ascending=False)

,clientes,churn_real_pct,churn_prob_media,revenue_en_riesgo
Segment_label,,,,
VIP,1594,6.6,24.9,237739.0
Dormidos,761,84.2,57.0,211217.0
En Riesgo,564,65.4,59.0,189491.0
Compradores de Sets,398,62.6,51.2,109864.0


In [9]:
res_ord = resumen.sort_values("revenue_en_riesgo", ascending=True)
fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(res_ord.index, res_ord["revenue_en_riesgo"] / 1000, color="salmon")
ax.set_xlabel("Revenue en riesgo (miles de £)")
ax.set_title("Revenue en riesgo por segmento (clientes con churn predicho)")
for bar, prob in zip(bars, res_ord["churn_prob_media"], strict=False):
    ax.text(
        bar.get_width(),
        bar.get_y() + bar.get_height() / 2,
        f"  prob. media {prob:.0f}%",
        va="center",
    )
plt.tight_layout()
plt.show()

/var/folders/80/chm8f4xj689g2c7r85cswh3m0000gn/T/ipykernel_78440/2814647343.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


> **Insight accionable:** combinando ambos modelos podemos priorizar la retención por **valor económico**, no solo por probabilidad. Un cliente VIP con riesgo medio puede representar más revenue en juego que varios clientes pequeños con riesgo alto. Esta tabla es, literalmente, una **lista de prioridades comerciales**.

## 6. Reflexión crítica

**Qué funciona bien**
- La segmentación produce grupos interpretables y alineados con el negocio (efecto Pareto claro).
- El modelo de churn alcanza un desempeño sólido y, sobre todo, es **interpretable**: indica qué variables mirar.
- El cruce segmento × churn convierte la predicción en una hoja de ruta de acción priorizada por valor.

**Limitaciones**
- El dataset cubre ~1 año de un único retailer del Reino Unido: hay estacionalidad y sesgo geográfico (UK domina el revenue).
- La definición de churn por ventana temporal es razonable pero sensible al punto de corte elegido.
- Aun tras la mejora del silhouette (~0.31), parte de la estructura de clientes es continua más que netamente separable: los límites entre segmentos contiguos no son tajantes.

**Mejora aplicada (feedback Entrega 03)**
- Subimos el silhouette de ~0.17 a ~0.31 (+80%) excluyendo `Cancel_rate` del clustering (feature ruidosa) y aplicando PCA antes de K-Means, sin perder la afinidad de producto ni los 4 segmentos de negocio.

**Posibles mejoras futuras**
- Calibrar el umbral de churn según el costo/beneficio real de cada acción de retención.
- Incorporar señales temporales (tendencia de compra) y un CLV estimado por cliente.
- Probar modelos de boosting con más datos y validar estabilidad de los segmentos en el tiempo.

## 7. Persistencia de los modelos

La consigna exige dejar los modelos **almacenados en un formato reutilizable** y **cargables sin reentrenar**. Guardamos cada modelo junto con su preprocesamiento y la lista de features, y luego lo recargamos para demostrar que predice sobre datos nuevos sin volver a entrenar.

In [10]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)

with (MODELS_DIR / "kmeans_model.pkl").open("wb") as f:
    pickle.dump({"model": kmeans, "scaler": preprocess, "features": FEATURES_CLUSTER}, f)

# Mapeo cluster -> etiqueta de negocio, consumido por la API REST de la Entrega 04.
with (MODELS_DIR / "cluster_labels.json").open("w", encoding="utf-8") as f:
    json.dump({str(c): lbl for c, lbl in labels.items()}, f, indent=2, ensure_ascii=False)

with (MODELS_DIR / "churn_model.pkl").open("wb") as f:
    pickle.dump(
        {"model": rf, "scaler": scaler, "features": FEATURE_NAMES, "model_name": "RandomForest"},
        f,
    )

print("Modelos guardados en:", MODELS_DIR.resolve())

# Demostración: recargar y predecir sin reentrenar
with (MODELS_DIR / "churn_model.pkl").open("rb") as f:
    bundle = pickle.load(f)  # noqa: S301 - artefacto propio versionado en el repo

sample = churn[bundle["features"]].iloc[[0]].to_numpy()
prob = bundle["model"].predict_proba(bundle["scaler"].transform(sample))[0, 1]
print(f"Modelo de churn recargado OK. Prob. de churn del cliente 0: {prob:.1%}")

Modelos guardados en: /Users/grc/Documents/CDDA/segmentacion-clientes-ecommerce/data/06_models


Modelo de churn recargado OK. Prob. de churn del cliente 0: 67.1%


## 8. Conclusiones y próximos pasos

- Construimos **dos modelos complementarios** que responden las preguntas de negocio: *quiénes son nuestros clientes* y *quiénes están por irse*.
- El verdadero valor surge al **combinarlos**: una lista priorizada de retención por revenue en riesgo.
- Los modelos quedan **persistidos y listos para usarse sin reentrenar**.

**Entrega 04 (próxima):** exponer estos modelos a través de un servicio/API e integrarlos en una interfaz de uso (prototipo Streamlit), más la presentación oral con demostración en vivo.